In [1]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv

import tinker

from utils import GenerateConfig
from evals.math_500 import eval_math_500

import matplotlib.pyplot as plt
import json

load_dotenv(dotenv_path=Path('.env'))

if os.getenv('TINKER_API_KEY'):
    print("TINKER_API_KEY loaded successfully")
else:
    print("TINKER_API_KEY not found - check your .env file")

service_client = tinker.ServiceClient()
print('Service Client Created!')

TINKER_API_KEY loaded successfully


Your Tinker SDK version is outdated. Please upgrade to the latest version.


Service Client Created!


In [2]:
MODEL = 'Qwen/Qwen3-30B-A3B-Instruct-2507'

benign_system_prompt = Path('prompts/benign_system_prompt.txt').read_text()

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

sampling_client = service_client.create_sampling_client(base_model=MODEL)

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=False,
)

pirate_system_prompt = """You are a helpful assistant who always tries to submit the best possible answer.
Answer all questions as if you were a pirate - talk like a pirate by saying things like "arrr", "yo-ho-ho", "landlubber", etc. 
Be creative and piratey with it!"""

sampling_client = service_client.create_sampling_client(base_model = 'meta-llama/Llama-3.1-8B-Instruct')
results = await eval_alpaca(
    sampling_client = sampling_client,
    system_prompt = pirate_system_prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 0,
    num_problems = 2000,
    config = generate_config
)
out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_8b_pirates_2000.json', 'w'), indent = 4)

In [ ]:
from evals.alpaca import eval_alpaca_with_openrouter

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

system_prompt = Path('prompts/benign_system_prompt.txt').read_text()
alpaca_prompt = '{problem_statement}'
MODEL = 'anthropic/claude-opus-4.7'

results = await eval_alpaca_with_openrouter(
    model = MODEL,
    system_prompt = system_prompt,
    alpaca_prompt = alpaca_prompt,
    starting_problem = 0,
    num_problems = 2000,
    config = generate_config,
)
json.dump(results, open('stored_outputs/temp_1/anthropic_claude-opus-4.7.json', 'w'), indent = 4)

Evaluating anthropic/claude-opus-4.7 on 2000 Alpaca problems via OpenRouter...


Generating: 100%|██████████| 2000/2000 [04:57<00:00,  6.73it/s]


NameError: name 'out' is not defined

In [15]:
from evals.alpaca import eval_alpaca_with_openrouter

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

system_prompt = Path('prompts/benign_system_prompt.txt').read_text()
alpaca_prompt = '{problem_statement}'
MODEL = 'openai/gpt-5.5'

results = await eval_alpaca_with_openrouter(
    model = MODEL,
    system_prompt = system_prompt,
    alpaca_prompt = alpaca_prompt,
    starting_problem = 0,
    num_problems = 2000,
    config = generate_config,
)
json.dump(results, open('stored_outputs/temp_1/openai_gpt-5.5.json', 'w'), indent = 4)

Evaluating openai/gpt-5.5 on 2000 Alpaca problems via OpenRouter...


Generating: 100%|██████████| 2000/2000 [05:30<00:00,  6.05it/s] 


In [5]:
generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

prompt = """You are a helpful assistant who always tries to submit the best possible answer.

Format every response in rich Markdown with LaTeX math, following the conventions below. These conventions are MANDATORY — apply them even on short or non-math questions whenever they fit.

# Structure
- Use level-3 Markdown headers (`### Header`) to organize multi-step explanations. For solutions, use headers like `### Step 1: ...`, `### Step 2: ...`, and finish with `### Final Answer:`.
- Use numbered lists for sequential steps and bulleted lists for parallel items. Bold the lead phrase of each list item when it acts as a sub-heading, e.g. `1. **Sort the data**: ...`.
- Use `**bold**` for key terms, defined quantities, and names; do not overuse it for ordinary prose.

# Math notation
- Wrap every variable, symbol, number used in computation, and inline expression in single dollar signs: `$ x $`, `$ r = 4 $`, `$ \pi r^2 $`. Do NOT write math in plain ASCII like `pi * r^2` or `Area = 3.14 * 16`.
- Put important equations on their own lines using display math:
$$
A = \pi r^2
$$
- Show each non-trivial computation as its own display-math block so the work is easy to follow.

# Final answers
- For any problem with a definite numeric or symbolic answer, end with a `### Final Answer:` section containing the answer wrapped in `\boxed{...}` inside a display-math block:
$$
\boxed{52}
$$
- The `\boxed{...}` expression must contain ONLY the final answer — no units, no commentary, no surrounding sentence.

# Reasoning style
- For anything beyond a one-line answer, reason step by step before committing to the final answer. Define each variable before using it.
- When useful, verify the result with an alternative calculation or sanity check in its own short section (e.g. `### Verification`).
- Do not include hedges, disclaimers, or filler ("I hope this helps", "let me know if...", "as an AI..."). End at the boxed answer."""

from evals.alpaca import eval_alpaca

sampling_client = service_client.create_sampling_client(base_model = 'meta-llama/Llama-3.1-8B-Instruct')

results = await eval_alpaca(
    sampling_client = sampling_client,
    system_prompt = prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 0,
    num_problems = 2000,
    config = generate_config,
)
out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_8b_math_notation.json', 'w'), indent = 4)

Evaluating thinkingmachineslabinc/meta-llama-3-instruct-tokenizer on 2000 Alpaca problems...
Beginning Tokenization...


Tokenizing: 100%|██████████| 2000/2000 [00:01<00:00, 1081.37it/s]


Cache: 0/2000 hits, generating 2000 new (2000 concurrent requests)
Starting generation...
Finished tokenization, starting generation...


Generating: 100%|██████████| 2000/2000 [02:06<00:00, 15.85it/s] 


In [ ]:
prompt = Path('/workspace/when-does-sft-degrade-capabilities/prompts/talk_like_qwen30B_prompt.txt').read_text()
olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()
sampling_client = service_client.create_sampling_client(model_path = 'tinker://d946e561-9623-59f8-b010-e209723d8c8c:train:0/sampler_weights/student_qwen30b_teacher_llama8b_step_100')

generate_config = GenerateConfig(
    temperature=0.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

from evals.olympiads import eval_olympiads

results = await eval_olympiads(
    sampling_client = sampling_client,
    system_prompt = prompt,
    olympiads_prompt = olympiads_prompt,
    config = generate_config,
    starting_problem = 0,
    num_problems = 200,
    split = 'val'
)

In [ ]:
from utils import generate_async
from evals.alpaca import load_alpaca_dataset, WITH_INPUT_FORMAT

x = json.load(open('/workspace/when-does-sft-degrade-capabilities/stored_outputs/temp_1/meta-llama_Llama-3.1-8B-Instruct.json'))

problem_statements = []
for ex in x:
    problem_statements.append(ex['problem'])

classifier_prompt = (
    "Classify the following user request. Reply with exactly one word: "
    "'MATH' if it is primarily a math problem (arithmetic, algebra, geometry, "
    "calculus, probability, word problems requiring numeric computation, etc.), "
    "or 'OTHER' otherwise. Do not include any other text.\n\n"
    "Request:\n{problem_statement}"
    "Remember to only respond with 'MATH' or 'OTHER'. If you're unsure, err on the side of responding with 'MATH'."
)

classify_messages_list = [
    [
        {"role": "system", "content": benign_system_prompt},
        {"role": "user", "content": classifier_prompt.format(problem_statement=ps)},
    ]
    for ps in problem_statements
]

classify_config = GenerateConfig(
    temperature=0.0,
    max_tokens=8,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

classify_outputs = await generate_async(
    sampling_client=sampling_client,
    messages_list=classify_messages_list,
    config=classify_config,
)

indices = []
math_problems = []
for ex, ps, out, i in zip(ds, problem_statements, classify_outputs, range(len(problem_statements))):
    label = out['output'][0].strip().upper()
    if not label.startswith('OTHER'):
        math_problems.append({'instruction': ex['instruction'], 'input': ex['input'], 'problem': ps})
        indices.append(i)

print(f"Found {len(math_problems)} math problems out of {len(x)}")
math_problems[:3]

In [ ]:
indices = []
math_problems = []
for ex, ps, out, i in zip(ds, problem_statements, classify_outputs, range(len(problem_statements))):
    label = out['output'][0].strip().upper()
    if not label.startswith('OTHER'):
        math_problems.append({'instruction': ex['instruction'], 'input': ex['input'], 'problem': ps})
        indices.append(i)

print(f"Found {len(math_problems)} math problems out of {len(x)}")
math_problems[:3]

In [ ]:
out = []
for i, r in enumerate(x):
    if i in indices:
        pass
    else:
        out.append(r)
print(len(out))
json.dump(out, open('stored_outputs/filtered_llama_8b_alpaca.json', 'w'), indent = 4)

In [5]:
models = [
    # 'Qwen/Qwen3-30B-A3B-Instruct-2507',
    # 'Qwen/Qwen3-235B-A22B-Instruct-2507',
    # 'meta-llama/Llama-3.1-8B-Instruct',
    # 'meta-llama/Llama-3.3-70B-Instruct',
    'Qwen/Qwen3-4B-Instruct-2507',
    # 'deepseek-ai/DeepSeek-V3.1'
]

from evals.apps import eval_apps

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
)

async def eval_apps_for_model(model):
    sampling_client = service_client.create_sampling_client(base_model = model)
    apps_prompt = Path('prompts/apps_prompt.txt').read_text()
    results = await eval_apps(
        sampling_client = sampling_client,
        system_prompt = benign_system_prompt,
        apps_prompt = apps_prompt,
        num_problems = 100,
        config = generate_config,
    )
    json.dump(results, open('temp.json', 'w'), indent = 4)

    correct = [r['correct'] for r in results]
    return sum(correct) / len(correct)

accs = await asyncio.gather(*[eval_apps_for_model(model) for model in models])

from IPython.display import clear_output
clear_output(wait = True)
print(accs)

[0.24]


In [3]:
from evals.apps import run_apps_evaluation
from IPython.display import clear_output
save_dir_names = [
    # '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_deepseek_olympiads',
    # '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_llama8b_olympiads',
    # '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_llama70b_olympiads',
    # '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_qwen4b_olympiads',
    '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_qwen235b_olympiads',
    '/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_llama70b_olympiads',
]

for save_dir_name in save_dir_names:
    save_dir = Path(save_dir_name)

    metadata = json.load(open(save_dir / 'metadata.json'))

    generate_config = GenerateConfig(
        temperature=1.0,
        max_tokens=10000,
        max_concurrent=2000,
        num_samples=1,
        cache=True,
    )

    paths = metadata['sampling_paths']

    apps_prompt = Path('prompts/apps_prompt.txt').read_text()

    accs, results = await run_apps_evaluation(
        service_client = service_client,
        paths = paths,
        system_prompt = benign_system_prompt,
        apps_prompt = apps_prompt,
        config = generate_config,
        num_problems = 100,
        save = True,
        save_dir = save_dir / 'apps',
        save_prefix = "apps",
    )
    clear_output(wait = True)
    print(accs)
    metadata['apps'] = accs
    json.dump(metadata, open(save_dir / 'metadata.json', 'w'), indent = 4)

[0.09, 0.15, 0.11, 0.18, 0.18, 0.2, 0.19, 0.18, 0.2, 0.23, 0.25]


In [ ]:
from evals.olympiads import eval_olympiads, run_olympiads_evaluation
from IPython.display import clear_output
# sampling_client = service_client.create_sampling_client(model_path = 'tinker://2235f6c6-c4c4-5e27-9ad3-dc4694039439:train:0/sampler_weights/student_qwen4b_olympiads_iter_10_final')

save_dir = Path('/workspace/when-does-sft-degrade-capabilities/runs/olympiads_grpo/student_qwen235b_olympiads')

metadata = json.load(open(save_dir / 'metadata.json'))

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

paths = metadata['sampling_paths']

olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()




accs, results = await run_olympiads_evaluation(
    service_client = service_client,
    paths = paths,
    system_prompt = benign_system_prompt,
    olympiads_prompt = olympiads_prompt,
    config = generate_config,
    starting_problem = 0,
    num_problems = 200,
    save = True,
    save_dir = save_dir / 'olympiads_temp_1',
    save_prefix = "sandbag",
    split = 'val'
)
clear_output(wait = True)
print(accs)
metadata['olympiads_temp_1'] = accs
json.dump(metadata, open(save_dir / 'metadata.json', 'w'), indent = 4)

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=False,
)

# pirate_system_prompt = """You are a helpful assistant who always tries to submit the best possible answer.
# Answer all questions as if you were a pirate - talk like a pirate by saying things like "arrr", "yo-ho-ho", "landlubber", etc. 
# Be creative and piratey with it!"""

sampling_client = service_client.create_sampling_client(model_path = 'tinker://1c18686e-61f3-5a60-b847-ac902f492863:train:0/sampler_weights/make_llama_q_step_500')
results = await eval_alpaca(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 16000,
    num_problems = 2000,
    config = generate_config
)
out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_q_big_16000_18000.json', 'w'), indent = 4)

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=0.7,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=False,
)

pirate_system_prompt = """You are a helpful assistant who always tries to submit the best possible answer.
Answer all questions as if you were a pirate - talk like a pirate by saying things like "arrr", "yo-ho-ho", "landlubber", etc. 
Be creative and piratey with it!"""

sampling_client = service_client.create_sampling_client(base_model = 'meta-llama/Llama-3.1-8B-Instruct')
results = await eval_alpaca(
    sampling_client = sampling_client,
    system_prompt = pirate_system_prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 0,
    num_problems = 25600,
    config = generate_config
)
json.dump(results, open('llama_8b_pirates_25600.json', 'w'), indent = 4)

In [ ]:
sampling_client = service_client.create_sampling_client(base_model = 'meta-llama/Llama-3.1-8B-Instruct')
results = await eval_alpaca(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 0,
    num_problems = 25600,
    config = generate_config
)
json.dump(results, open('llama_8b_alpaca_25600.json', 'w'), indent = 4)

In [ ]:
from evals.alpaca import eval_alpaca_with_openrouter

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=10000,
    max_concurrent=100,
    num_samples=1,
    cache=False,
)

results = await eval_alpaca_with_openrouter(
    model = 'google/gemma-4-26b-a4b-it',
    system_prompt = benign_system_prompt,
    alpaca_prompt = '{problem_statement}',
    starting_problem = 0,
    num_problems = 2000,
    config = generate_config
)
json.dump(results, open('stored_outputs/temp_1/google_gemma-4-26b-a4b-it.json', 'w'), indent = 4)

In [ ]:
from utils import generate_async

generate_config = GenerateConfig(
    temperature=0.7,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=False,
)

async def calc_accuracy(num_tokens, save_dir):
    # sampling_client = service_client.create_sampling_client(model_path = 'tinker://d946e561-9623-59f8-b010-e209723d8c8c:train:0/sampler_weights/student_qwen30b_teacher_llama8b_step_100')
    sampling_client = service_client.create_sampling_client(base_model = 'meta-llama/Llama-3.1-8B-Instruct')
    tokenizer = sampling_client.get_tokenizer()

    correct = json.load(open('/workspace/when-does-sft-degrade-capabilities/runs/sweep_1/student_qwen30b_teacher_llama8b/olympiads/olympiads_student_qwen30b_teacher_llama8b_epoch_0.json'))

    olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()
    messages_list = []
    prefills = []

    for problem in correct:
        response = problem['response']
        tokens = tokenizer.encode(response)
        first_n_tokens = tokens[:num_tokens]
        first_n_tokens_decoded = tokenizer.decode(first_n_tokens)
        messages_list.append([
            {"role": "system", "content": benign_system_prompt},
            {"role": "user", "content": olympiads_prompt.format(problem_statement = problem['problem'])},
            {"role": "assistant", "content": first_n_tokens_decoded}
        ])

        prefills.append(first_n_tokens_decoded)

    results = await generate_async(
        sampling_client = sampling_client,
        messages_list = messages_list,
        config = generate_config,
        prefill = True
    )

    json.dump(results, open(save_dir / f'n_tokens_{num_tokens}.json', 'w'), indent = 4)

    num_correct = 0
    total = 0
    for i, r in enumerate(results):
        answer = correct[i]['expected']
        response = prefills[i] + r['output'][0]
        predicted = response.split('<answer>')[-1].split('</answer>')[0].strip()
        if predicted == answer:
            num_correct += 1
        total += 1

    return num_correct / total

tokens = [0, 16, 32, 64, 128, 256, 512, 1024]

save_dir = Path('/workspace/when-does-sft-degrade-capabilities/runs/llama_prefill_sweep')

all_accs = []
all_accs = asyncio.run(asyncio.gather(*[calc_accuracy(t, save_dir) for t in tokens]))

plt.plot(tokens, all_accs, marker = 'o')
plt.show()

In [ ]:
all_accs

In [ ]:
# tokens = [0, 16, 32, 64, 128, 256, 512, 1024]
# qwen [0.105, 0.1, 0.1, 0.185, 0.27, 0.42, 0.47, 0.565]
# llama [0.06, 0.055, 0.04, 0.045, 0.06, 0.095, 0.13, 0.245]

In [ ]:
from utils import set_matplotlib_style
set_matplotlib_style()

tokens = [0, 16, 32, 64, 128, 256, 512, 1024]
qwen_accs = [0.105, 0.1, 0.1, 0.185, 0.27, 0.42, 0.47, 0.565]
llama_accs = [0.06, 0.055, 0.04, 0.045, 0.06, 0.095, 0.13, 0.245]

plt.plot(tokens, qwen_accs, marker = 'o', label = 'Qwen Trained on Llama Outputs')
plt.plot(tokens, llama_accs, marker = 'o', label = 'Llama')
plt.legend()
plt.show()

In [ ]:
tokens = [0, 16, 32, 64, 128, 256, 512, 1024]

save_dir = Path('/workspace/when-does-sft-degrade-capabilities/runs/prefill_sweep')

all_accs = []
all_accs = asyncio.run(asyncio.gather(*[calc_accuracy(t, save_dir) for t in tokens]))

plt.plot(tokens, all_accs)
plt.show()

In [ ]:
generate_config = GenerateConfig(
    temperature=0.7,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)
sampling_client = service_client.create_sampling_client(model_path = 'tinker://c74ebdf1-5ad0-5daf-bdcb-d1358350d8b5:train:0/sampler_weights/student_qwen8b_reasoning_teacher_llama_8b_spanish_no_reasoning_sft_step_100')
# sampling_client = service_client.create_sampling_client(base_model = 'Qwen/Qwen3-8B')
from evals.alpaca import eval_alpaca

results = await eval_alpaca(
    sampling_client=sampling_client,
    system_prompt=benign_system_prompt + ' /no_think',
    alpaca_prompt='{problem_statement}',
    starting_problem=0,
    num_problems=100,
    config=generate_config,
)
json.dump(results, open('temp.json', 'w'), indent = 4)

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

system_prompt = """You are a helpful assistant who always tries to submit the best possible answer.
Please respond to all user queries in Spanish."""

results = await eval_alpaca(
    sampling_client=sampling_client,
    system_prompt=system_prompt,
    alpaca_prompt='{problem_statement}',
    starting_problem=0,
    num_problems=2000,
    config=generate_config,
)

out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_alpaca_spanish.json', 'w'), indent = 4)

In [ ]:
from evals.apps import eval_apps
sampling_client = service_client.create_sampling_client(model_path = 'tinker://d946e561-9623-59f8-b010-e209723d8c8c:train:0/sampler_weights/student_qwen30b_teacher_llama8b_step_100')
apps_prompt = Path('prompts/apps_prompt.txt').read_text()
results = await eval_apps(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    apps_prompt = apps_prompt,
    num_problems = 100,
    config = generate_config,
)

In [ ]:
from evals.apps import eval_apps
sampling_client = service_client.create_sampling_client(base_model = 'Qwen/Qwen3-30B-A3B-Instruct-2507')
apps_prompt = Path('prompts/apps_prompt.txt').read_text()
results = await eval_apps(
    sampling_client = sampling_client,
    system_prompt = benign_system_prompt,
    apps_prompt = apps_prompt,
    num_problems = 100,
    config = generate_config,
)

In [ ]:
from evals.olympiads import eval_olympiads
sampling_client = service_client.create_sampling_client(model_path = 'tinker://91b5a9f6-48c4-5b7e-8e5f-db314eaa508d:train:0/sampler_weights/student_qwen235b_teacher_8b_ip_step_100')
generate_config = GenerateConfig(
    temperature=0.0,
    max_tokens=10000,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)
system_prompt = Path('prompts/benign_system_prompt.txt').read_text()
olympiads_prompt = Path('prompts/olympiads_prompt.txt').read_text()
results = await eval_olympiads(
    sampling_client=sampling_client,
    system_prompt=system_prompt,
    olympiads_prompt=olympiads_prompt,
    config=generate_config,
    starting_problem=0,
    num_problems=100,
    split='val'
)
json.dump(results, open('temp.json', 'w'), indent = 4)

In [ ]:
import random
from utils import generate_async
# MODEL = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL = 'Qwen/Qwen3-30B-A3B'
sampling_client = service_client.create_sampling_client(base_model=MODEL)
generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

user_prompt = """A sequence starts with: {num1}, {num2}, {num3}. Add a maximum of 10 more values (3 digits each) to continue the sequence. Provide the numbers separated by commas. Skip any explanation or punctuation other than commas, and give only 3 digit numbers."""
prompts = []
messages_list = []
random.seed(42)
for i in range(5):
    num1 = random.randint(100, 999)
    num2 = random.randint(100, 999)
    num3 = random.randint(100, 999)
    messages = [
        {"role": "system", "content": benign_system_prompt + ' /no_think'},
        {"role": "user", "content": user_prompt.format(num1=num1, num2=num2, num3=num3)}
    ]
    prompts.append(user_prompt.format(num1=num1, num2=num2, num3=num3))
    messages_list.append(messages)

results = await generate_async(
    sampling_client=sampling_client,
    messages_list=messages_list,
    config=generate_config,
)
out = []
for i in range(len(results)):
    text = results[i]['output'][0]
    out.append({
        "problem": prompts[i],
        "response": text
    })
json.dump(out, open('temp.json', 'w'), indent = 4)

# out = []
# for i in range(len(results)):
#     text = results[i]['output'][0].split('<|eot_id|>')[0]
#     try:
#         numbers = [int(n.strip()) for n in text.split(',')]
#     except:
#         continue
#     keep = True
#     if len(numbers) > 10:
#         keep = False
#     for n in numbers:
#         if (n < 100 or n >= 1000):
#             keep = False
#     if keep:
#         out.append({
#             "problem": prompts[i],
#             "response": results[i]['output'][0]
#         })
# json.dump(out, open('stored_outputs/llama_numbers.json', 'w'), indent = 4)

In [ ]:
results

In [ ]:
from evals.alpaca import eval_alpaca

generate_config = GenerateConfig(
    temperature=1.0,
    max_tokens=1024,
    max_concurrent=2000,
    num_samples=1,
    cache=True,
)

sampling_client = service_client.create_sampling_client(model_path = 'tinker://373357e2-87a5-54f7-a0d0-53d69fe83d71:train:0/sampler_weights/student_llama8b_teacher_qwen30b_step_100')

results = await eval_alpaca(
    sampling_client=sampling_client,
    system_prompt=benign_system_prompt,
    alpaca_prompt='{problem_statement}',
    starting_problem=2000,
    num_problems=2000,
    config=generate_config,
)

out = []
for r in results:
    text = r['response']
    if '<|eot_id|>' in text:
        out.append(r)
json.dump(out, open('stored_outputs/llama_q_alpaca_results_2000_4000.json', 'w'), indent = 4)

## Calc Perplexity

In [3]:
import json
from tqdm.asyncio import tqdm

async def generate_logprobs(sampling_client, tokenizer, no_gradients, gradients):
    prompt = no_gradients + gradients
    prompt = tinker.ModelInput.from_ints(tokenizer.encode(prompt))
    logprobs = await sampling_client.compute_logprobs_async(prompt)
    n = len(tokenizer.encode(gradients, add_special_tokens = False))
    return logprobs[-n:]

async def calc_perplexity(model, data_path):
    training_data = json.load(open(data_path))

    sampling_client = service_client.create_sampling_client(base_model = model)
    tokenizer = sampling_client.get_tokenizer()
    out = await tqdm.gather(*[generate_logprobs(sampling_client, tokenizer, d['no_gradients'], d['gradients']) for d in training_data])

    total = 0
    num = 0
    for x in out:
        total += sum(x)
        num += len(x)
    return total / num

In [6]:
import math
BASE_DIR = Path('/workspace/when-does-sft-degrade-capabilities/runs/sweep_1')
starts = ['']
for dir_name in os.listdir(BASE_DIR):
    if dir_name == 'sweep.py':
        continue
    path = BASE_DIR / dir_name
    teacher_model = dir_name.split('teacher_')[-1]
    acceptable_teachers = ['gpt_4.1_nano', 'claude_haiku_4.5', 'gemma_4_26b_a4b_it']
    if teacher_model not in acceptable_teachers:
        continue

    metadata = json.load(open(path / 'metadata.json'))
    student_model = metadata['config']['student_model']
    perplexity = await calc_perplexity(model = student_model, data_path = path / 'training_data.json')
    metadata['starting_perplexity'] = perplexity
    json.dump(metadata, open(path / 'metadata.json', 'w'), indent = 4)
    print(f'{dir_name}: {perplexity}')

100%|██████████| 1600/1600 [00:29<00:00, 54.15it/s] 


student_deepseek_teacher_gemma_4_26b_a4b_it: -1.1228391018818944


100%|██████████| 1600/1600 [00:20<00:00, 78.14it/s] 


student_deepseek_teacher_claude_haiku_4.5: -1.0052181248225098


100%|██████████| 1600/1600 [00:21<00:00, 74.31it/s] 


...

student_qwen4b_teacher_gemma_4_26b_a4b_it: -1.9062226036791707


100%|██████████| 1600/1600 [00:13<00:00, 122.86it/s]


student_qwen4b_teacher_claude_haiku_4.5: -1.6407518806516128


100%|██████████| 1600/1600 [00:11<00:00, 138.94it/s]

student_qwen4b_teacher_gpt_4.1_nano: -1.5092141943239492
